# KTMD Kin2 / Su corrected montage: background + progress dashboard

この版は、長時間の解析をColabセルの中で待ち続けません。解析を**バックグラウンドプロセス**として起動し、起動セルは数秒で終了します。

- 長時間セルのタイムアウトを回避
- 30秒ごとにGoogle Driveへheartbeatと進捗を保存
- 6実験日それぞれの進捗を表示
- 45分以上ログ・出力更新がない場合にstall warning
- runtimeが切れてもRun allでcheckpointから再開
- 二重起動を防止

出力フォルダの `PROGRESS/` に `LATEST_STATUS.md`, `status.html`, `progress.json`, `progress.csv`, `heartbeat.txt` が更新されます。

**ランタイム → すべてのセルを実行**してください。asset ZIPは従来と同じものを使います。


In [ ]:
from google.colab import drive
import subprocess, sys

drive.mount("/content/drive", force_remount=False)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "networkx", "gdown", "scikit-learn"],
    check=True,
)
print("Drive mounted and dependencies installed.")


In [ ]:
from pathlib import Path
import hashlib, shutil, sys, zipfile
from google.colab import files

ASSET_NAME = "KTMD_corrected_fullrerun_asset_v4_submission.zip"
EXPECTED_ASSET_SHA256 = "861bb97c0fe44f05506ee42bb3e027956d399bb7dd3ded1ba94f95e48cbf4969"
DRIVE_ASSET = Path("/content/drive/MyDrive/Global Workspace Analysis/KTMD_corrected_fullrerun_asset_v4_submission.zip")
LOCAL_ASSET = Path("/content") / ASSET_NAME
ASSET_DIR = Path("/content/KTMD_corrected_fullrerun_asset_v4_submission")

if DRIVE_ASSET.exists():
    asset_zip = DRIVE_ASSET
    print("Using persistent Drive asset:", asset_zip)
elif LOCAL_ASSET.exists():
    asset_zip = LOCAL_ASSET
    print("Using asset already uploaded to this runtime:", asset_zip)
else:
    print("Select the sidecar file:", ASSET_NAME)
    uploaded = files.upload()
    if ASSET_NAME not in uploaded:
        raise RuntimeError(f"Please select {ASSET_NAME}. Uploaded: {list(uploaded)}")
    LOCAL_ASSET.write_bytes(uploaded[ASSET_NAME])
    asset_zip = LOCAL_ASSET

observed_sha = hashlib.sha256(asset_zip.read_bytes()).hexdigest()
if observed_sha != EXPECTED_ASSET_SHA256:
    raise RuntimeError(f"Asset SHA mismatch: expected {EXPECTED_ASSET_SHA256}, observed {observed_sha}")
if asset_zip != DRIVE_ASSET:
    DRIVE_ASSET.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(asset_zip, DRIVE_ASSET)
    print("Saved persistent asset to:", DRIVE_ASSET)
if ASSET_DIR.exists():
    shutil.rmtree(ASSET_DIR)
ASSET_DIR.mkdir(parents=True)
with zipfile.ZipFile(asset_zip) as zf:
    bad = zf.testzip()
    if bad:
        raise RuntimeError(f"Asset ZIP is corrupt at {bad}")
    zf.extractall(ASSET_DIR)
sys.path.insert(0, str(ASSET_DIR))
print("Assets extracted:", ASSET_DIR)


In [ ]:
from pathlib import Path
import shutil, gdown
from run_full_corrected_pipeline import TARGETS, load_expected, find_manifest_and_parts

PUBLIC_SPLIT_FOLDER = "https://drive.google.com/drive/folders/1j8BZzWSOpinykXxf7W63HPxgJquiHgEU?usp=drive_link"
EXPECTED = load_expected(ASSET_DIR)
CANDIDATES = [
    Path("/content/drive/MyDrive/Global Workspace Analysis/NeuroTycho_Monkey_ECoG_Split80MiB"),
    Path("/content/drive/MyDrive/Global Workspace Analysis/NeuroTycho_Monkey_ECoG_Downloads"),
    Path("/content/drive/MyDrive/Global Workspace Analysis/NeuroTycho_Monkey_ECoG_Downloads/NeuroTycho_Monkey_ECoG_Split80MiB"),
]

def recognized_targets(root: Path) -> int:
    if not root.exists(): return 0
    count = 0
    for animal, date in TARGETS:
        name = f"{date}KTMD_Anesthesia+and+Sleep_{animal}_Toru+Yanagawa_mat_ECoG128.zip"
        try:
            find_manifest_and_parts(root, animal, date, EXPECTED[name])
            count += 1
        except Exception:
            pass
    return count

SPLIT_ROOT = None
for candidate in CANDIDATES:
    n = recognized_targets(candidate)
    print(f"Input audit: {candidate} -> {n}/6 target archives")
    if n == 6:
        SPLIT_ROOT = candidate
        break
if SPLIT_ROOT is None:
    PUBLIC_CACHE = Path("/content/ktmd_shared_parts")
    if PUBLIC_CACHE.exists() and recognized_targets(PUBLIC_CACHE) < 6:
        shutil.rmtree(PUBLIC_CACHE)
    PUBLIC_CACHE.mkdir(parents=True, exist_ok=True)
    print("Six split archives not found in MyDrive; downloading public shared folder...")
    gdown.download_folder(url=PUBLIC_SPLIT_FOLDER, output=str(PUBLIC_CACHE), quiet=False,
                          use_cookies=False, remaining_ok=True)
    n = recognized_targets(PUBLIC_CACHE)
    print(f"Public-folder audit: {n}/6 target archives")
    if n != 6:
        raise RuntimeError("Public-folder download did not expose all six complete manifests/parts.")
    SPLIT_ROOT = PUBLIC_CACHE
print("Verified split root:", SPLIT_ROOT)


In [ ]:
from pathlib import Path
import py_compile
SUPERVISOR_PATH = Path("/content/ktmd_progress_supervisor.py")
SUPERVISOR_SOURCE = "".join([
    '#!/usr/bin/env python3\n',
    '"""Background supervisor and persistent Drive dashboard for the KTMD rerun."""\n',
    'from __future__ import annotations\n',
    '\n',
    'import argparse\n',
    'import csv\n',
    'import html\n',
    'import json\n',
    'import os\n',
    'import selectors\n',
    'import signal\n',
    'import subprocess\n',
    'import sys\n',
    'import time\n',
    'import traceback\n',
    'from collections import deque\n',
    'from datetime import datetime, timezone\n',
    'from pathlib import Path\n',
    'from typing import Any, Iterable\n',
    '\n',
    'TARGETS = (\n',
    '    ("Kin2", "20110513"), ("Kin2", "20110524"), ("Kin2", "20110525"),\n',
    '    ("Su", "20110523"), ("Su", "20110526"), ("Su", "20110527"),\n',
    ')\n',
    'DAY_CHECKS = (\n',
    '    ("raw archive verified", ("RAW_ARCHIVE_PROVENANCE.json",), .12),\n',
    '    ("preprocessing complete", ("PREPROCESS_COMPLETE.ok",), .28),\n',
    '    ("state model fitted", ("model_fit.csv",), .48),\n',
    '    ("candidate search complete", ("candidate_sets.csv", "candidate_signatures.csv"), .68),\n',
    '    ("within-day cross-fit complete", ("crossfit_ratios.csv",), .88),\n',
    '    ("day complete", ("DAY_COMPLETE.ok", "summary.json"), 1.0),\n',
    ')\n',
    'GLOBAL_CHECKS = (\n',
    '    ("corrected LODO", ("*leave_one_day_out*", "*lodo*ratio*"), .20),\n',
    '    ("11-day within-day table", ("updated_within_day_awake_crossfit_ratios_11days.csv",), .35),\n',
    '    ("11-day LODO table", ("updated_same_animal_leave_one_day_out_ratios.csv",), .50),\n',
    '    ("animal-balanced hierarchy", ("updated_hierarchical_lodo_deep_effects.csv",), .65),\n',
    '    ("recovery and slow-wave controls", ("*recovery*.csv", "*slow*wave*.csv"), .76),\n',
    '    ("manuscript figures", ("figure_updated_crossday_transfer.png", "figure_updated_subject_specific_official_maps.png"), .86),\n',
    '    ("writing-thread materials", ("RESULTS_FOR_WRITING_THREAD.json", "MANUSCRIPT_RESULTS_PATCH.md"), .94),\n',
    '    ("strict final audit", ("ALL_ANALYSES_COMPLETE.ok", "REAL_RAW_DATA_PROVENANCE_VERIFIED.ok"), 1.0),\n',
    ')\n',
    '\n',
    '\n',
    'def utc_now() -> str:\n',
    '    return datetime.now(timezone.utc).isoformat(timespec="seconds")\n',
    '\n',
    '\n',
    'def atomic_write(path: Path, text: str) -> None:\n',
    '    path.parent.mkdir(parents=True, exist_ok=True)\n',
    '    tmp = path.with_suffix(path.suffix + ".tmp")\n',
    '    tmp.write_text(text, encoding="utf-8")\n',
    '    os.replace(tmp, path)\n',
    '\n',
    '\n',
    'def atomic_json(path: Path, obj: Any) -> None:\n',
    '    atomic_write(path, json.dumps(obj, ensure_ascii=False, indent=2, default=str))\n',
    '\n',
    '\n',
    'def pid_alive(pid: int | None) -> bool:\n',
    '    if not pid or pid <= 0:\n',
    '        return False\n',
    '    try:\n',
    '        os.kill(pid, 0)\n',
    '        return True\n',
    '    except OSError:\n',
    '        return False\n',
    '\n',
    '\n',
    'def read_json(path: Path) -> dict[str, Any]:\n',
    '    try:\n',
    '        obj = json.loads(path.read_text(encoding="utf-8"))\n',
    '        return obj if isinstance(obj, dict) else {}\n',
    '    except Exception:\n',
    '        return {}\n',
    '\n',
    '\n',
    'def read_tail(path: Path, n: int = 50) -> list[str]:\n',
    '    lines: deque[str] = deque(maxlen=n)\n',
    '    if not path.exists():\n',
    '        return []\n',
    '    try:\n',
    '        with path.open("r", encoding="utf-8", errors="replace") as fh:\n',
    '            for line in fh:\n',
    '                lines.append(line.rstrip("\\n"))\n',
    '    except Exception as exc:\n',
    '        return [f"Could not read log: {exc}"]\n',
    '    return list(lines)\n',
    '\n',
    '\n',
    'def all_files(roots: Iterable[Path]) -> list[Path]:\n',
    '    result: list[Path] = []\n',
    '    for root in roots:\n',
    '        if root.exists():\n',
    '            try:\n',
    '                result.extend(p for p in root.rglob("*") if p.is_file())\n',
    '            except Exception:\n',
    '                pass\n',
    '    return result\n',
    '\n',
    '\n',
    'def summary_identity(path: Path) -> tuple[str | None, str | None]:\n',
    '    if path.name != "summary.json":\n',
    '        return None, None\n',
    '    data = read_json(path)\n',
    '    a, d = data.get("animal"), data.get("date")\n',
    '    return (str(a) if a is not None else None, str(d) if d is not None else None)\n',
    '\n',
    '\n',
    'def files_for_day(files: list[Path], animal: str, date: str) -> list[Path]:\n',
    '    selected = [p for p in files if animal.lower() in str(p).lower() and date in str(p)]\n',
    '    for summary in (p for p in files if p.name == "summary.json"):\n',
    '        a, d = summary_identity(summary)\n',
    '        if a == animal and d == date:\n',
    '            parent = summary.parent\n',
    '            selected.extend(p for p in files if p == parent or parent in p.parents)\n',
    '    return list(dict.fromkeys(selected))\n',
    '\n',
    '\n',
    'def stage_check(day_files: list[Path], names: tuple[str, ...]) -> bool:\n',
    '    available = {p.name for p in day_files}\n',
    '    if names == ("DAY_COMPLETE.ok", "summary.json"):\n',
    '        return "DAY_COMPLETE.ok" in available or (\n',
    '            "summary.json" in available and "candidate_sets.csv" in available and "crossfit_ratios.csv" in available\n',
    '        )\n',
    '    return all(name in available for name in names)\n',
    '\n',
    '\n',
    'def day_status(files: list[Path], animal: str, date: str) -> dict[str, Any]:\n',
    '    subset = files_for_day(files, animal, date)\n',
    '    fraction, stage = 0.0, "not started"\n',
    '    checks: dict[str, bool] = {}\n',
    '    for label, names, value in DAY_CHECKS:\n',
    '        ok = stage_check(subset, names)\n',
    '        checks[label] = ok\n',
    '        if ok and value >= fraction:\n',
    '            fraction, stage = value, label\n',
    '    newest = max((p.stat().st_mtime for p in subset), default=0.0)\n',
    '    return {\n',
    '        "animal": animal, "date": date, "fraction": fraction,\n',
    '        "percent": round(100 * fraction, 1), "stage": stage, "checks": checks,\n',
    '        "file_count": len(subset),\n',
    '        "last_update_utc": datetime.fromtimestamp(newest, timezone.utc).isoformat(timespec="seconds") if newest else None,\n',
    '    }\n',
    '\n',
    '\n',
    'def glob_any(root: Path, pattern: str) -> bool:\n',
    '    try:\n',
    '        return root.exists() and any(root.rglob(pattern))\n',
    '    except Exception:\n',
    '        return False\n',
    '\n',
    '\n',
    'def downstream_status(output_root: Path) -> dict[str, Any]:\n',
    '    fraction, stage = 0.0, "waiting for six day-level analyses"\n',
    '    checks: dict[str, bool] = {}\n',
    '    for label, patterns, value in GLOBAL_CHECKS:\n',
    '        ok = all(glob_any(output_root, pattern) for pattern in patterns)\n',
    '        checks[label] = ok\n',
    '        if ok and value >= fraction:\n',
    '            fraction, stage = value, label\n',
    '    return {"fraction": fraction, "percent": round(100 * fraction, 1), "stage": stage, "checks": checks}\n',
    '\n',
    '\n',
    'def progress_snapshot(*, output_root: Path, work_root: Path, progress_dir: Path, log_path: Path,\n',
    '                      supervisor_pid: int, child_pid: int | None, started_at: float,\n',
    '                      return_code: int | None, attempt: int, stall_minutes: float) -> dict[str, Any]:\n',
    '    files = all_files([output_root, work_root])\n',
    '    days = [day_status(files, a, d) for a, d in TARGETS]\n',
    '    day_fraction = sum(d["fraction"] for d in days) / len(days)\n',
    '    downstream = downstream_status(output_root)\n',
    '    overall = min(1.0, .78 * day_fraction + .22 * downstream["fraction"])\n',
    '    mtimes = [p.stat().st_mtime for p in files]\n',
    '    if log_path.exists():\n',
    '        mtimes.append(log_path.stat().st_mtime)\n',
    '    last_activity = max(mtimes, default=time.time())\n',
    '    idle_seconds = max(0.0, time.time() - last_activity)\n',
    '    running = pid_alive(child_pid) if return_code is None else False\n',
    '    marker = (output_root / "ALL_ANALYSES_COMPLETE.ok").exists()\n',
    '    if return_code is None and running:\n',
    '        state = "running"\n',
    '    elif return_code == 0 and marker:\n',
    '        state = "complete"\n',
    '    elif return_code == 0:\n',
    '        state = "finished_incomplete"\n',
    '    elif return_code is None:\n',
    '        state = "stopped_or_stale"\n',
    '    else:\n',
    '        state = "failed"\n',
    '    tail = read_tail(log_path)\n',
    '    current = next((line for line in reversed(tail) if line.strip() and "[HEARTBEAT]" not in line), "No pipeline console output yet")\n',
    '    return {\n',
    '        "updated_utc": utc_now(), "state": state, "attempt": attempt,\n',
    '        "supervisor_pid": supervisor_pid, "pipeline_pid": child_pid,\n',
    '        "pipeline_pid_alive": running, "return_code": return_code,\n',
    '        "started_utc": datetime.fromtimestamp(started_at, timezone.utc).isoformat(timespec="seconds"),\n',
    '        "elapsed_minutes": round((time.time() - started_at) / 60, 1),\n',
    '        "overall_percent": round(100 * overall, 1),\n',
    '        "completed_days": sum(d["fraction"] >= 1.0 for d in days), "target_days": len(days),\n',
    '        "current_message": current[-800:],\n',
    '        "last_activity_utc": datetime.fromtimestamp(last_activity, timezone.utc).isoformat(timespec="seconds"),\n',
    '        "idle_minutes": round(idle_seconds / 60, 1),\n',
    '        "stall_warning": bool(running and idle_seconds > stall_minutes * 60),\n',
    '        "days": days, "downstream": downstream,\n',
    '        "paths": {"output_root": str(output_root), "work_root": str(work_root),\n',
    '                  "progress_dir": str(progress_dir), "console_log": str(log_path)},\n',
    '        "log_tail": tail,\n',
    '    }\n',
    '\n',
    '\n',
    'def publish(progress_dir: Path, data: dict[str, Any], refresh_seconds: int) -> None:\n',
    '    progress_dir.mkdir(parents=True, exist_ok=True)\n',
    '    atomic_json(progress_dir / "progress.json", data)\n',
    '    csv_tmp = progress_dir / "progress.csv.tmp"\n',
    '    with csv_tmp.open("w", newline="", encoding="utf-8") as fh:\n',
    '        writer = csv.DictWriter(fh, fieldnames=["animal", "date", "percent", "stage", "file_count", "last_update_utc"])\n',
    '        writer.writeheader()\n',
    '        for row in data["days"]:\n',
    '            writer.writerow({key: row.get(key) for key in writer.fieldnames})\n',
    '    os.replace(csv_tmp, progress_dir / "progress.csv")\n',
    '\n',
    '    md = [\n',
    '        "# KTMD corrected-montage rerun progress", "",\n',
    '        f"- Updated: **{data[\'updated_utc\']}**", f"- State: **{data[\'state\']}**",\n',
    '        f"- Overall: **{data[\'overall_percent\']:.1f}%**",\n',
    '        f"- Completed days: **{data[\'completed_days\']}/{data[\'target_days\']}**",\n',
    '        f"- Elapsed: **{data[\'elapsed_minutes\']:.1f} min**",\n',
    '        f"- Last activity: **{data[\'last_activity_utc\']}** ({data[\'idle_minutes\']:.1f} min ago)",\n',
    '        f"- Current message: `{data[\'current_message\']}`", "",\n',
    '        "| Animal | Date | Progress | Stage | Last update |", "|---|---:|---:|---|---|",\n',
    '    ]\n',
    '    for day in data["days"]:\n',
    '        md.append(f"| {day[\'animal\']} | {day[\'date\']} | {day[\'percent\']:.1f}% | {day[\'stage\']} | {day[\'last_update_utc\'] or \'—\'} |")\n',
    '    md += ["", "## Downstream aggregation", ""]\n',
    '    md += [f"- {\'✅\' if ok else \'⬜\'} {label}" for label, ok in data["downstream"]["checks"].items()]\n',
    '    if data["stall_warning"]:\n',
    '        md += ["", "> **Warning:** no log or output-file update beyond the stall threshold."]\n',
    '    atomic_write(progress_dir / "LATEST_STATUS.md", "\\n".join(md) + "\\n")\n',
    '\n',
    '    rows = "".join(\n',
    '        f"<tr><td>{html.escape(d[\'animal\'])}</td><td>{d[\'date\']}</td><td>{d[\'percent\']:.1f}%</td>"\n',
    '        f"<td>{html.escape(d[\'stage\'])}</td><td>{html.escape(d[\'last_update_utc\'] or \'—\')}</td></tr>"\n',
    '        for d in data["days"]\n',
    '    )\n',
    '    checklist = "".join(f"<li>{\'✅\' if ok else \'⬜\'} {html.escape(label)}</li>" for label, ok in data["downstream"]["checks"].items())\n',
    '    warning = "<div class=\'warning\'>No log or output-file change beyond the stall threshold.</div>" if data["stall_warning"] else ""\n',
    '    log_tail = html.escape("\\n".join(data["log_tail"]))\n',
    '    pct = data["overall_percent"]\n',
    '    page = f"""<!doctype html><html><head><meta charset=\'utf-8\'><meta http-equiv=\'refresh\' content=\'{refresh_seconds}\'>\n',
    '<title>KTMD progress</title><style>\n',
    'body{{font-family:Arial,sans-serif;max-width:1100px;margin:24px auto;padding:0 18px;line-height:1.45}}\n',
    '.bar{{height:26px;background:#e5e7eb;border-radius:13px;overflow:hidden;border:1px solid #aaa}}\n',
    '.fill{{height:100%;width:{pct:.1f}%;background:linear-gradient(90deg,#fbbf24,#dc2626)}}\n',
    'table{{border-collapse:collapse;width:100%;margin:16px 0}}th,td{{border:1px solid #bbb;padding:7px}}th{{background:#f3f4f6}}\n',
    'pre{{background:#111827;color:#e5e7eb;padding:12px;overflow:auto;max-height:430px;white-space:pre-wrap}}\n',
    '.warning{{background:#fee2e2;border-left:5px solid #b91c1c;padding:10px;margin:12px 0}}\n',
    '.small{{color:#555;font-size:.92em}}</style></head><body>\n',
    "<h1>KTMD corrected-montage full rerun</h1><div class='bar'><div class='fill'></div></div>\n",
    "<h2>{pct:.1f}% — {html.escape(data['state'])}</h2>\n",
    "<p>Completed days: <b>{data['completed_days']}/{data['target_days']}</b> · Elapsed: <b>{data['elapsed_minutes']:.1f} min</b> · Attempt: <b>{data['attempt']}</b></p>\n",
    "<p>Current message: <code>{html.escape(data['current_message'])}</code></p>\n",
    "<p class='small'>Updated {data['updated_utc']}; last activity {data['last_activity_utc']} ({data['idle_minutes']:.1f} min ago). Refreshes every {refresh_seconds}s.</p>{warning}\n",
    '<h2>Six corrected days</h2><table><tr><th>Animal</th><th>Date</th><th>Progress</th><th>Stage</th><th>Last update</th></tr>{rows}</table>\n',
    '<h2>Downstream analyses</h2><ul>{checklist}</ul><h2>Recent console output</h2><pre>{log_tail}</pre></body></html>"""\n',
    '    atomic_write(progress_dir / "status.html", page)\n',
    '    atomic_write(progress_dir / "heartbeat.txt", f"{data[\'updated_utc\']}\\t{data[\'state\']}\\t{pct:.1f}%\\t{data[\'completed_days\']}/{data[\'target_days\']} days\\t{data[\'current_message\']}\\n")\n',
    '\n',
    '\n',
    'def run(args: argparse.Namespace) -> int:\n',
    '    progress_dir = args.progress_dir\n',
    '    progress_dir.mkdir(parents=True, exist_ok=True)\n',
    '    old = read_json(progress_dir / "run_state.json")\n',
    '    if (pid_alive(int(old.get("supervisor_pid") or 0)) or pid_alive(int(old.get("pipeline_pid") or 0))) and not args.force:\n',
    '        print("A rerun process is already alive; use status or stop.", file=sys.stderr)\n',
    '        return 2\n',
    '    attempt = int(old.get("attempt") or 0) + 1\n',
    '    started, supervisor_pid = time.time(), os.getpid()\n',
    '    state = {"attempt": attempt, "supervisor_pid": supervisor_pid, "pipeline_pid": None,\n',
    '             "started_utc": utc_now(), "command": args.pipeline_cmd, "state": "starting"}\n',
    '    atomic_json(progress_dir / "run_state.json", state)\n',
    '    env = os.environ.copy()\n',
    '    env.update({"PYTHONUNBUFFERED": "1", "MPLBACKEND": "Agg", "OMP_NUM_THREADS": str(args.threads),\n',
    '                "MKL_NUM_THREADS": str(args.threads), "OPENBLAS_NUM_THREADS": str(args.threads),\n',
    '                "NUMEXPR_NUM_THREADS": str(args.threads)})\n',
    '    return_code: int | None = None\n',
    '    child: subprocess.Popen[str] | None = None\n',
    '    try:\n',
    '        args.log_path.parent.mkdir(parents=True, exist_ok=True)\n',
    '        with args.log_path.open("a", encoding="utf-8", buffering=1) as log:\n',
    '            log.write(f"\\n\\n=== SUPERVISED RUN ATTEMPT {attempt} @ {utc_now()} ===\\nCOMMAND: {\' \'.join(args.pipeline_cmd)}\\n")\n',
    '            child = subprocess.Popen(args.pipeline_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,\n',
    '                                     text=True, bufsize=1, env=env, start_new_session=True)\n',
    '            state.update({"pipeline_pid": child.pid, "state": "running"})\n',
    '            atomic_json(progress_dir / "run_state.json", state)\n',
    '            assert child.stdout is not None\n',
    '            selector = selectors.DefaultSelector()\n',
    '            selector.register(child.stdout, selectors.EVENT_READ)\n',
    '            next_heartbeat = 0.0\n',
    '            while True:\n',
    '                events = selector.select(timeout=1.0)\n',
    '                line = child.stdout.readline() if events else ""\n',
    '                if line:\n',
    '                    log.write(f"[{utc_now()}] {line.rstrip()}\\n")\n',
    '                polled, now = child.poll(), time.time()\n',
    '                if now >= next_heartbeat or polled is not None:\n',
    '                    data = progress_snapshot(output_root=args.output_root, work_root=args.work_root,\n',
    '                                             progress_dir=progress_dir, log_path=args.log_path,\n',
    '                                             supervisor_pid=supervisor_pid, child_pid=child.pid,\n',
    '                                             started_at=started, return_code=polled, attempt=attempt,\n',
    '                                             stall_minutes=args.stall_minutes)\n',
    '                    publish(progress_dir, data, args.refresh_seconds)\n',
    '                    log.write(f"[{utc_now()}] [HEARTBEAT] {data[\'overall_percent\']:.1f}% · {data[\'completed_days\']}/{data[\'target_days\']} days · idle {data[\'idle_minutes\']:.1f} min\\n")\n',
    '                    next_heartbeat = now + args.heartbeat_seconds\n',
    '                if polled is not None:\n',
    '                    return_code = int(polled)\n',
    '                    break\n',
    '    except BaseException:\n',
    '        return_code = 130 if isinstance(sys.exc_info()[1], KeyboardInterrupt) else 1\n',
    '        with args.log_path.open("a", encoding="utf-8") as log:\n',
    '            log.write(f"[{utc_now()}] SUPERVISOR EXCEPTION\\n{traceback.format_exc()}\\n")\n',
    '        if child and child.poll() is None:\n',
    '            try: os.killpg(child.pid, signal.SIGTERM)\n',
    '            except Exception: pass\n',
    '    finally:\n',
    '        pid = child.pid if child else None\n',
    '        data = progress_snapshot(output_root=args.output_root, work_root=args.work_root,\n',
    '                                 progress_dir=progress_dir, log_path=args.log_path,\n',
    '                                 supervisor_pid=supervisor_pid, child_pid=pid,\n',
    '                                 started_at=started, return_code=return_code, attempt=attempt,\n',
    '                                 stall_minutes=args.stall_minutes)\n',
    '        publish(progress_dir, data, args.refresh_seconds)\n',
    '        state.update({"state": data["state"], "return_code": return_code, "finished_utc": utc_now(), "pipeline_pid": pid})\n',
    '        atomic_json(progress_dir / "run_state.json", state)\n',
    '        atomic_write(progress_dir / "EXIT_CODE.txt", f"{return_code}\\n")\n',
    '    return int(return_code or 0)\n',
    '\n',
    '\n',
    'def status(args: argparse.Namespace) -> int:\n',
    '    data = read_json(args.progress_dir / "progress.json")\n',
    '    if not data:\n',
    '        print("No progress snapshot exists yet.")\n',
    '        return 1\n',
    '    print(f"{data.get(\'state\')} · {data.get(\'overall_percent\',0):.1f}% · {data.get(\'completed_days\',0)}/{data.get(\'target_days\',6)} days · elapsed {data.get(\'elapsed_minutes\',0):.1f} min")\n',
    '    for d in data.get("days", []):\n',
    '        print(f"  {d[\'animal\']} {d[\'date\']}: {d[\'percent\']:5.1f}%  {d[\'stage\']}")\n',
    '    print("Current:", data.get("current_message"))\n',
    '    if data.get("stall_warning"): print("WARNING: stall threshold exceeded.")\n',
    '    print("Status HTML:", args.progress_dir / "status.html")\n',
    '    print("Markdown:", args.progress_dir / "LATEST_STATUS.md")\n',
    '    print("Log:", data.get("paths", {}).get("console_log"))\n',
    '    return 0\n',
    '\n',
    '\n',
    'def stop(args: argparse.Namespace) -> int:\n',
    '    state = read_json(args.progress_dir / "run_state.json")\n',
    '    stopped = False\n',
    '    for key in ("pipeline_pid", "supervisor_pid"):\n',
    '        pid = int(state.get(key) or 0)\n',
    '        if pid_alive(pid):\n',
    '            try: os.killpg(pid, signal.SIGTERM)\n',
    '            except ProcessLookupError: continue\n',
    '            except PermissionError: os.kill(pid, signal.SIGTERM)\n',
    '            print("Sent SIGTERM to", pid)\n',
    '            stopped = True\n',
    '    return 0 if stopped else 1\n',
    '\n',
    '\n',
    'def build_parser() -> argparse.ArgumentParser:\n',
    '    parser = argparse.ArgumentParser(description=__doc__)\n',
    '    sub = parser.add_subparsers(dest="command", required=True)\n',
    '    p = sub.add_parser("run")\n',
    '    p.add_argument("--output-root", type=Path, required=True)\n',
    '    p.add_argument("--work-root", type=Path, required=True)\n',
    '    p.add_argument("--progress-dir", type=Path, required=True)\n',
    '    p.add_argument("--log-path", type=Path, required=True)\n',
    '    p.add_argument("--heartbeat-seconds", type=int, default=30)\n',
    '    p.add_argument("--refresh-seconds", type=int, default=30)\n',
    '    p.add_argument("--stall-minutes", type=float, default=45)\n',
    '    p.add_argument("--threads", type=int, default=2)\n',
    '    p.add_argument("--force", action="store_true")\n',
    '    p.add_argument("pipeline_cmd", nargs=argparse.REMAINDER)\n',
    '    p = sub.add_parser("status"); p.add_argument("--progress-dir", type=Path, required=True)\n',
    '    p = sub.add_parser("stop"); p.add_argument("--progress-dir", type=Path, required=True)\n',
    '    return parser\n',
    '\n',
    '\n',
    'def main() -> int:\n',
    '    args = build_parser().parse_args()\n',
    '    if args.command == "run":\n',
    '        if args.pipeline_cmd and args.pipeline_cmd[0] == "--": args.pipeline_cmd = args.pipeline_cmd[1:]\n',
    '        if not args.pipeline_cmd: raise SystemExit("Pass pipeline command after --")\n',
    '        return run(args)\n',
    '    if args.command == "status": return status(args)\n',
    '    if args.command == "stop": return stop(args)\n',
    '    return 2\n',
    '\n',
    '\n',
    'if __name__ == "__main__":\n',
    '    raise SystemExit(main())\n',
])
SUPERVISOR_PATH.write_text(SUPERVISOR_SOURCE, encoding="utf-8")
py_compile.compile(str(SUPERVISOR_PATH), doraise=True)
print("Progress supervisor installed:", SUPERVISOR_PATH)


In [ ]:
from pathlib import Path
import json, os, subprocess, sys, time

OUTPUT_ROOT = Path("/content/drive/MyDrive/Global Workspace Analysis/KTMD_Kin2_Su_CorrectedMontage_FullRerun_SubmissionReady_20260816")
WORK_ROOT = Path("/content/ktmd_corrected_fullrerun_work")
PROGRESS_DIR = OUTPUT_ROOT / "PROGRESS"
LOG_PATH = OUTPUT_ROOT / "FULL_RERUN_CONSOLE.log"
LAUNCHER_LOG = PROGRESS_DIR / "SUPERVISOR_LAUNCHER.log"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
PROGRESS_DIR.mkdir(parents=True, exist_ok=True)

# Prevent accidental duplicate analysis if the old synchronous notebook is still running.
existing = subprocess.run(
    ["pgrep", "-af", "run_full_corrected_pipeline.py"],
    capture_output=True,
    text=True,
    check=False,
).stdout.strip()
if existing:
    raise RuntimeError(
        "An older full-rerun pipeline is still running in this runtime. "
        "Stop that cell or restart the Colab runtime once, then run this notebook again.\n"
        + existing
    )

pipeline_cmd = [sys.executable, "-u", str(ASSET_DIR / "run_full_corrected_pipeline.py"),
                "--split-root", str(SPLIT_ROOT), "--output-root", str(OUTPUT_ROOT),
                "--assets", str(ASSET_DIR), "--work-root", str(WORK_ROOT)]
supervisor_cmd = [sys.executable, str(SUPERVISOR_PATH), "run",
                  "--output-root", str(OUTPUT_ROOT), "--work-root", str(WORK_ROOT),
                  "--progress-dir", str(PROGRESS_DIR), "--log-path", str(LOG_PATH),
                  "--heartbeat-seconds", "30", "--refresh-seconds", "30",
                  "--stall-minutes", "45", "--threads", "2", "--", *pipeline_cmd]

print("Launching background pipeline:", " ".join(pipeline_cmd))
print("Progress:", PROGRESS_DIR)
with LAUNCHER_LOG.open("a", encoding="utf-8") as fh:
    fh.write("\\n=== LAUNCH REQUEST ===\\n" + " ".join(supervisor_cmd) + "\\n")
    fh.flush()
    launcher = subprocess.Popen(
        supervisor_cmd,
        stdout=fh,
        stderr=subprocess.STDOUT,
        start_new_session=True,
        env={**os.environ, "PYTHONUNBUFFERED": "1"},
    )
print("Background supervisor PID:", launcher.pid)
time.sleep(4)
state_path = PROGRESS_DIR / "run_state.json"
print(
    json.dumps(json.loads(state_path.read_text()), indent=2)
    if state_path.exists()
    else "Supervisor is starting; re-run the status cell shortly."
)
print("This cell is finished. The analysis continues in the background.")


## 進捗確認（何度でも再実行可能）


In [ ]:
import subprocess, sys
from IPython.display import HTML, display

print("ONE-SHOT STATUS — re-run this cell whenever you want an update\\n")
subprocess.run(
    [sys.executable, str(SUPERVISOR_PATH), "status", "--progress-dir", str(PROGRESS_DIR)],
    check=False,
)
status_html = PROGRESS_DIR / "status.html"
if status_html.exists():
    display(HTML(status_html.read_text(encoding="utf-8")))
else:
    print("No dashboard yet. Wait about 30 seconds and re-run this cell.")
print("\\nDrive-visible files:")
for name in ["LATEST_STATUS.md", "status.html", "progress.json", "progress.csv", "heartbeat.txt"]:
    print(" -", PROGRESS_DIR / name)


## 任意の停止・再開


In [ ]:
# OPTIONAL: set ACTION to status, stop, or restart.
import os, subprocess, sys, time
ACTION = "status"
if ACTION == "status":
    subprocess.run([sys.executable, str(SUPERVISOR_PATH), "status", "--progress-dir", str(PROGRESS_DIR)], check=False)
elif ACTION == "stop":
    subprocess.run([sys.executable, str(SUPERVISOR_PATH), "stop", "--progress-dir", str(PROGRESS_DIR)], check=False)
elif ACTION == "restart":
    with LAUNCHER_LOG.open("a", encoding="utf-8") as fh:
        subprocess.Popen(supervisor_cmd, stdout=fh, stderr=subprocess.STDOUT,
                         start_new_session=True, env={**os.environ, "PYTHONUNBUFFERED": "1"})
    time.sleep(4)
    print("Restart requested. Re-run the status cell.")
else:
    raise ValueError("ACTION must be status, stop, or restart")


## 完了後の最終監査


In [ ]:
import json, pandas as pd, subprocess, sys
completion = OUTPUT_ROOT / "ALL_ANALYSES_COMPLETE.ok"
provenance = OUTPUT_ROOT / "REAL_RAW_DATA_PROVENANCE_VERIFIED.ok"
if not (completion.exists() and provenance.exists()):
    print("The full rerun is not complete yet. Check:", PROGRESS_DIR / "LATEST_STATUS.md")
else:
    required = {
        "completion marker": completion,
        "raw provenance marker": provenance,
        "raw provenance table": OUTPUT_ROOT / "tables/RAW_ARCHIVE_PROVENANCE_ALL_SIX_DAYS.csv",
        "completion audit": OUTPUT_ROOT / "COMPLETION_AUDIT.json",
        "final summary": OUTPUT_ROOT / "FINAL_SUMMARY.json",
        "updated LODO table": OUTPUT_ROOT / "tables/updated_hierarchical_lodo_deep_effects.csv",
        "updated subject maps": OUTPUT_ROOT / "figures/figure_updated_subject_specific_official_maps.png",
        "updated cross-day figure": OUTPUT_ROOT / "figures/figure_updated_crossday_transfer.png",
        "writing-thread JSON": OUTPUT_ROOT / "manuscript_materials/RESULTS_FOR_WRITING_THREAD.json",
        "Results patch": OUTPUT_ROOT / "manuscript_materials/MANUSCRIPT_RESULTS_PATCH.md",
    }
    for label, path in required.items(): print("[OK]" if path.exists() else "[MISSING]", label, path)
    if not all(path.exists() for path in required.values()):
        raise RuntimeError("Required manuscript outputs are missing")
    audit = json.loads((OUTPUT_ROOT / "COMPLETION_AUDIT.json").read_text())
    if not audit.get("all_required_complete", False): raise RuntimeError(f"Strict audit failed: {audit}")
    print(json.dumps(audit, indent=2))
    display(pd.read_csv(OUTPUT_ROOT / "tables/updated_hierarchical_lodo_deep_effects.csv"))
    subprocess.run([sys.executable, str(ASSET_DIR / "strict_postrun_verifier.py"), str(OUTPUT_ROOT)], check=True)
    DERIVED_ZIP = OUTPUT_ROOT.parent / (OUTPUT_ROOT.name + "_DERIVED_RESULTS.zip")
    print("ALL SIX CORRECTED RERUNS COMPLETE — RAW PROVENANCE LOCKED")
    print("Compact manuscript handoff ZIP:", DERIVED_ZIP)
